In [1]:
import os
# import jupytergis
# _tiler_src = "/home/greg/Documents/programming/quantstack/jupytergis-tiler/src/jupytergis"
# if _tiler_src not in jupytergis.__path__:
# 	jupytergis.__path__.append(_tiler_src)
# os.environ["AWS_VIRTUAL_HOSTING"] = "FALSE"
# os.environ["AWS_ENDPOINT_URL"] = "https://eodata.dataspace.copernicus.eu"
# os.environ["AWS_ACCESS_KEY_ID"] = "HGWDKN4N26XVYNYMRXP0"
# os.environ["AWS_SECRET_ACCESS_KEY"] = "R893NB8q4vTK1NaZ7BH6Aa5Yq043O6zJ9lHE96sn"
os.environ["AWS_ACCESS_KEY_ID"] = "ZJS870ZUZKFVGGP0STGL"
os.environ["AWS_SECRET_ACCESS_KEY"] = "hct8Qmx0n6e41ZK92y5EsKCrX4u8QCXYAcdr3CE4"
# os.environ["AWS_DEFAULT_REGION"] = "default"   
# os.environ["AWS_REGION"] = "eu-central-1"
# if that still gives signature errors, try:
# os.environ["AWS_REGION"] = "us-east-1"


os.environ["GDAL_HTTP_TCP_KEEPALIVE"] = "YES"
os.environ["AWS_S3_ENDPOINT"] = "eodata.dataspace.copernicus.eu"
# os.environ["AWS_ACCESS_KEY_ID"] = os.environ.get("CDSE_S3_ACCESS_KEY")
# os.environ["AWS_SECRET_ACCESS_KEY"] = os.environ.get("CDSE_S3_SECRET_KEY")
os.environ["AWS_HTTPS"] = "YES"
os.environ["AWS_VIRTUAL_HOSTING"] = "FALSE"
os.environ["GDAL_HTTP_UNSAFESSL"] = "YES"




In [2]:
import stackstac

from jupytergis.tiler import GISDocument

doc = GISDocument()

doc.add_raster_layer(
    url="https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}",
    name="Google Satellite",
    attribution="Google",
    opacity=0.6,
)

doc

In [3]:
item = doc.stacItem

In [5]:
item

[{'id': 'S2A_MSIL2A_20260308T110841_N0512_R137_T30TWT_20260308T175910',
  'bbox': [-3.0002673576574,
   46.85663988190739,
   -1.532722623231765,
   47.85370184248753],
  'type': 'Feature',
  'links': [{'rel': 'collection',
    'type': 'application/json',
    'href': 'https://stac.dataspace.copernicus.eu/v1/collections/sentinel-2-l2a'},
   {'rel': 'parent',
    'type': 'application/json',
    'href': 'https://stac.dataspace.copernicus.eu/v1/collections/sentinel-2-l2a'},
   {'rel': 'root',
    'type': 'application/json',
    'href': 'https://stac.dataspace.copernicus.eu/v1/'},
   {'rel': 'self',
    'type': 'application/geo+json',
    'href': 'https://stac.dataspace.copernicus.eu/v1/collections/sentinel-2-l2a/items/S2A_MSIL2A_20260308T110841_N0512_R137_T30TWT_20260308T175910'},
   {'rel': 'version-history',
    'href': 'https://trace.dataspace.copernicus.eu/api/v1/traces/name/S2A_MSIL2A_20260308T110841_N0512_R137_T30TWT_20260308T175910.SAFE.zip',
    'type': 'application/json',
    'tit

In [4]:
da = stackstac.stack(item, epsg=3857)

In [5]:
green = da.sel(band='B03_10m')
nir = da.sel(band='B08_10m')
ndwi = (green - nir) / (green + nir)
ndwi = ndwi.where((green + nir) != 0)
ndwi.name = 'ndwi'

In [6]:
import dask.diagnostics
with dask.diagnostics.ProgressBar():
    data = ndwi.compute()

[########################################] | 100% Completed | 67.42 s


In [7]:
vmin_acc, vmax_acc = int(data.min().compute()), int(data.max().compute())
vmin_acc, vmax_acc

(0, 1)

In [8]:
out = data.isel(time=0)

In [ ]:
await doc.add_tiler_layer(
    name="ndwi",
    data_array=out,
    colormap_name="plasma",
    rescale=(vmin_acc, vmax_acc),
    reproject="max",
)

[2026-03-30 11:24:27 +0200] [198165] [ERROR] Error in ASGI Framework
Traceback (most recent call last):
  File "/home/greg/.local/share/mamba/envs/jgis-normal/lib/python3.14/site-packages/anycorn/task_group.py", line 38, in _handle
    await app(scope, receive, send, sync_spawn, call_soon)
  File "/home/greg/.local/share/mamba/envs/jgis-normal/lib/python3.14/site-packages/anycorn/app_wrappers.py", line 42, in __call__
    await self.app(scope, receive, send)
  File "/home/greg/.local/share/mamba/envs/jgis-normal/lib/python3.14/site-packages/fastapi/applications.py", line 1160, in __call__
    await super().__call__(scope, receive, send)
  File "/home/greg/.local/share/mamba/envs/jgis-normal/lib/python3.14/site-packages/starlette/applications.py", line 90, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/home/greg/.local/share/mamba/envs/jgis-normal/lib/python3.14/site-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/home/g